# Baseline XResNet1d50 — same train rows as skewed-sex SSSD generator

Trains **XResNet1d50** on **real PTB-XL only** (same setup as `Train_custum_SSSD_ECG.ipynb` cells 53–55), but the **training subset** is defined **only** by the row indices saved in `Train_custom_SSSD_ECG_on_skewed_data.ipynb`:

- `generator_train_row_indices_seed{SUBSET_SEED}.npy` under `DRIVE_SUBSET_DIR` (same files used by `Generate_ECGs_on_skewed_sex.ipynb` for `ptbxl_train_data_seed*.npy`).
- Val/test folds are the standard PTB-XL last folds, identical to the other notebooks.

**Reproducibility:** use the same raw PTB-XL extract and preprocessing (`prepare_data_ptb_xl` -> memmap) used to build the subset; otherwise row ids may not align with signals.

**Config:** set `SUBSET_SEED` and `DRIVE_SUBSET_DIR` to match the skewed training run.

**Training seeds:** `TRAIN_SEEDS` (default **three** runs: `[0, 1, 2]`) fixes PyTorch/NumPy/Python RNG and the train `DataLoader` shuffle **for the same row subset**; the notebook reports **mean ± std** over test AUROC/AUPRC and sex-stratified AUROC/AUPRC.

**Real + synthetic:** loops **`ALPHAS`** and **`N_MIX_SEEDS`** train seeds; reports **mean ± std per α** plus tables. Loads four `*_data.npy` / `*_cond.npy` groups, same batches/epoch as baseline. Set **`VERBOSE_MIX_EPOCHS`** for per-epoch loss logs.


In [ ]:
!pip install -q pykeops

In [ ]:
# Environment + repo (same pattern as Train_custum / skewed SSSD notebooks)
import os
import sys
from pathlib import Path

!pip install -q wfdb resampy ishneholterlib pytorch-lightning

REPO_URL = "https://github.com/Anonymous0002396/CMED-Mini-Project.git"
PROJECT_ROOT = Path("/content/CMED-Mini-Project")
REPO_ROOT = PROJECT_ROOT / "SSSD-ECG"

if not PROJECT_ROOT.exists():
 !git clone {REPO_URL} /content/CMED-Mini-Project
else:
 %cd /content/CMED-Mini-Project
 !git pull

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "src" / "ptb_xl"))
sys.path.insert(0, str(REPO_ROOT / "src" / "sssd"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_ROOT:", REPO_ROOT)

In [ ]:
# Mount Drive + PTB-XL raw data (same as Train_custom_SSSD_ECG_on_skewed_data.ipynb)
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

if not os.path.exists("/content/ptbxl.zip"):
 !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip

if not os.path.exists("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1"):
 !unzip -oq /content/ptbxl.zip -d /content/
 print("Extraction complete.")
else:
 print("PTB-XL already extracted.")

raw_ptbxl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
print("raw_ptbxl exists:", raw_ptbxl.exists())

In [ ]:
# Preprocess PTB-XL @ 100 Hz (same as other notebooks in this project)
import numpy as np
from pathlib import Path

from clinical_ts.ecg_utils import prepare_data_ptb_xl, channel_stoi_default
from clinical_ts.timeseries_utils import reformat_as_memmap, load_dataset

target_fs = 100
data_folder_ptb_xl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
target_folder_ptb_xl = Path(f"/content/processed_ptb_xl_fs{target_fs}")

if target_folder_ptb_xl.exists():
 !rm -rf {target_folder_ptb_xl}

print("Rebuilding processed PTB-XL at:", target_folder_ptb_xl)

df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
 data_path=data_folder_ptb_xl,
 min_cnt=0,
 target_fs=target_fs,
 channels=12,
 channel_stoi=channel_stoi_default,
 target_folder=target_folder_ptb_xl,
 recreate_data=True,
)

reformat_as_memmap(
 df_ptb_xl,
 target_folder_ptb_xl / "memmap.npy",
 data_folder=target_folder_ptb_xl,
 delete_npys=True,
)

df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)
print("Processed df shape:", df_mapped.shape)
print("Processed folder:", target_folder_ptb_xl)

In [ ]:
# Sex + binary MI labels (same SCP-derived MI rule as Train_custum_SSSD_ECG.ipynb)
import numpy as np
import pandas as pd

label_key = "label_all"
label_names = np.array(lbl_itos_dict[label_key])

mi_statement_names = [
 "IMI", "ASMI", "ILMI", "AMI", "ALMI",
 "INJAS", "LMI", "INJAL", "IPLMI", "IPMI",
 "INJIN", "PMI", "INJLA", "INJIL",
]

label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]

def multihot_encode(indices, num_classes):
 out = np.zeros(num_classes, dtype=np.float32)
 for i in indices:
 out[i] = 1.0
 return out

def row_multihot_to_binary_mi(row_vec, mi_indices):
 return float(row_vec[mi_indices].sum() > 0)

df_real = df_mapped.copy()
df_real["label_multihot"] = df_real[f"{label_key}_numeric"].apply(
 lambda x: multihot_encode(x, len(label_names))
)
df_real["label_mi"] = df_real["label_multihot"].apply(
 lambda x: row_multihot_to_binary_mi(x, mi_label_indices)
).astype(np.float32)
df_real["sex_binary"] = df_real["sex"].astype(np.float32)

print(df_real.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# Memmap metadata + conditioning vector [sex, MI]
import pickle
import numpy as np

with open(target_folder_ptb_xl / "df_memmap.pkl", "rb") as f:
 df_memmap_meta = pickle.load(f)

df_memmap_meta = df_memmap_meta.copy()
df_memmap_meta["sex_binary"] = df_real["sex_binary"].values.astype(np.float32)
df_memmap_meta["label_mi"] = df_real["label_mi"].values.astype(np.float32)
df_memmap_meta["strat_fold"] = df_real["strat_fold"].values

cond_matrix = np.stack(
 [
 df_memmap_meta["sex_binary"].to_numpy(dtype=np.float32),
 df_memmap_meta["label_mi"].to_numpy(dtype=np.float32),
 ],
 axis=1,
).astype(np.float32)
df_memmap_meta["cond_sex_mi"] = list(cond_matrix)

In [ ]:
# --- Match Train_custom_SSSD_ECG_on_skewed_data.ipynb / Generate_ECGs_on_skewed_sex.ipynb ---
SUBSET_SEED = 42
DRIVE_SUBSET_DIR = Path("/content/drive/MyDrive/sssd_sex_mi_skewed_25f_75m_subsets")

idx_path = DRIVE_SUBSET_DIR / f"generator_train_row_indices_seed{SUBSET_SEED}.npy"
if not idx_path.exists():
 raise FileNotFoundError(
 f"Missing {idx_path}. Run the skewed SSSD notebook cell that saves indices, "
 "or copy the file and set DRIVE_SUBSET_DIR / SUBSET_SEED."
 )

row_indices = np.load(idx_path)
missing = np.setdiff1d(row_indices, df_memmap_meta.index.to_numpy())
if len(missing):
 raise ValueError(
 f"{len(missing)} indices from {idx_path.name} are not in df_memmap_meta — "
 "raw/preprocessed PTB-XL likely differs from the subset run."
 )

max_fold_id = df_memmap_meta["strat_fold"].max()
df_val_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id - 1].copy()
df_test_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id].copy()

# Same rows as generator / materialized ptbxl_train_data_seed*.npy (order preserved)
df_train_real_mem_subset = df_memmap_meta.loc[row_indices].copy()

print("Train subset:", len(df_train_real_mem_subset))
print("Val:", len(df_val_real_mem), "| Test:", len(df_test_real_mem))
print("Female fraction (train):", (df_train_real_mem_subset["sex_binary"] == 1.0).mean())
print(df_train_real_mem_subset.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# Optional: verify memmap iteration matches saved NPY from the skewed run
ref_npy = DRIVE_SUBSET_DIR / f"ptbxl_train_data_seed{SUBSET_SEED}.npy"
if ref_npy.exists():
 from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor

 _tfms = ToTensor()
 _ds = TimeseriesDatasetCrops(
 df_train_real_mem_subset,
 output_size=1000,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=1000,
 stride=1000,
 transforms=_tfms,
 annotation=False,
 col_lbl="cond_sex_mi",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
 )
 xs = np.stack([_ds[i][0].numpy().astype(np.float32) for i in range(len(_ds))])
 ref = np.load(ref_npy, mmap_mode="r")
 print("max |memmap_stack - ptbxl_train_data_seed|.:", float(np.max(np.abs(xs - ref))))
else:
 print("No reference file at", ref_npy, "— skip numeric check.")

In [ ]:
# Datasets + DataLoaders (real train only — baseline)
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor

input_size = 1000
tfms_ptb_xl = ToTensor()

train_real_ds_full12_subset = TimeseriesDatasetCrops(
 df_train_real_mem_subset,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="cond_sex_mi",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

val_real_ds_full12 = TimeseriesDatasetCrops(
 df_val_real_mem,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="cond_sex_mi",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

test_real_ds_full12 = TimeseriesDatasetCrops(
 df_test_real_mem,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="cond_sex_mi",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

class TupleToDictWrapper(Dataset):
 def __init__(self, base_ds):
 self.base_ds = base_ds

 def __len__(self):
 return len(self.base_ds)

 def __getitem__(self, idx):
 x, y = self.base_ds[idx]
 x = x.float() if torch.is_tensor(x) else torch.tensor(x, dtype=torch.float32)
 y = y.float() if torch.is_tensor(y) else torch.tensor(y, dtype=torch.float32)
 return {
 "signal": x,
 "label": y[1],
 "cond": y,
 }

train_real_ds = TupleToDictWrapper(train_real_ds_full12_subset)
val_real_ds = TupleToDictWrapper(val_real_ds_full12)
test_real_ds = TupleToDictWrapper(test_real_ds_full12)

batch_size = 64

def make_train_real_loader(train_seed: int) -> DataLoader:
 g = torch.Generator()
 g.manual_seed(int(train_seed))
 return DataLoader(
 train_real_ds,
 batch_size=batch_size,
 shuffle=True,
 num_workers=2,
 pin_memory=True,
 generator=g,
 )

val_loader = DataLoader(
 val_real_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
 test_real_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)

print("train:", len(train_real_ds), "val:", len(val_real_ds), "test:", len(test_real_ds))

In [ ]:
# Training loop helpers (same as Train_custum_SSSD_ECG.ipynb cell used for XResNet)
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()

def train_one_epoch(model, loader, optimizer, device):
 model.train()
 total_loss, n = 0.0, 0
 for batch in loader:
 x = batch["signal"].to(device, non_blocking=True)
 y = batch["label"].to(device, non_blocking=True).float()
 optimizer.zero_grad()
 logits = model(x)
 loss = criterion(logits, y)
 loss.backward()
 optimizer.step()
 bs = x.size(0)
 total_loss += loss.item() * bs
 n += bs
 return total_loss / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, device):
 model.eval()
 total_loss, n = 0.0, 0
 all_probs, all_targets = [], []
 for batch in loader:
 x = batch["signal"].to(device, non_blocking=True)
 y = batch["label"].to(device, non_blocking=True).float()
 logits = model(x)
 loss = criterion(logits, y)
 probs = torch.sigmoid(logits)
 bs = x.size(0)
 total_loss += loss.item() * bs
 n += bs
 all_probs.append(probs.cpu())
 all_targets.append(y.cpu())
 all_probs = torch.cat(all_probs).numpy()
 all_targets = torch.cat(all_targets).numpy()
 return {
 "loss": total_loss / max(n, 1),
 "auroc": roc_auc_score(all_targets, all_probs),
 "auprc": average_precision_score(all_targets, all_probs),
 "probs": all_probs,
 "targets": all_targets,
 }

def fit_model(model, train_loader, val_loader, device, epochs=20, lr=1e-3, verbose=True, log_prefix=""):
 optimizer = torch.optim.Adam(model.parameters(), lr=lr)
 best_state = None
 best_val_auroc = -np.inf
 for epoch in range(1, epochs + 1):
 train_loss = train_one_epoch(model, train_loader, optimizer, device)
 val_metrics = evaluate(model, val_loader, device)
 if verbose:
 print(
 f"{log_prefix}Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | "
 f"val_loss={val_metrics['loss']:.4f} | val_auroc={val_metrics['auroc']:.4f} | "
 f"val_auprc={val_metrics['auprc']:.4f}"
 )
 if val_metrics["auroc"] > best_val_auroc:
 best_val_auroc = val_metrics["auroc"]
 best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
 model.load_state_dict(best_state)
 return model

In [ ]:
# XResNet1d50 (same architecture as Train_custum_SSSD_ECG.ipynb)
import torch
import torch.nn as nn

class Bottleneck1D(nn.Module):
 expansion = 4

 def __init__(self, in_channels, planes, stride=1):
 super().__init__()
 out_channels = planes * self.expansion
 self.conv1 = nn.Conv1d(in_channels, planes, kernel_size=1, bias=False)
 self.bn1 = nn.BatchNorm1d(planes)
 self.conv2 = nn.Conv1d(
 planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
 )
 self.bn2 = nn.BatchNorm1d(planes)
 self.conv3 = nn.Conv1d(planes, out_channels, kernel_size=1, bias=False)
 self.bn3 = nn.BatchNorm1d(out_channels)
 self.relu = nn.ReLU(inplace=True)
 self.downsample = None
 if stride != 1 or in_channels != out_channels:
 self.downsample = nn.Sequential(
 nn.Conv1d(
 in_channels, out_channels, kernel_size=1, stride=stride, bias=False
 ),
 nn.BatchNorm1d(out_channels),
 )

 def forward(self, x):
 identity = x
 out = self.conv1(x)
 out = self.bn1(out)
 out = self.relu(out)
 out = self.conv2(out)
 out = self.bn2(out)
 out = self.relu(out)
 out = self.conv3(out)
 out = self.bn3(out)
 if self.downsample is not None:
 identity = self.downsample(x)
 out = out + identity
 out = self.relu(out)
 return out

class XResNet1d50BinaryMIClassifier(nn.Module):
 def __init__(self, in_channels=12, base_filters=64, num_classes=1):
 super().__init__()
 self.stem = nn.Sequential(
 nn.Conv1d(
 in_channels, 32, kernel_size=5, stride=2, padding=2, bias=False
 ),
 nn.BatchNorm1d(32),
 nn.ReLU(inplace=True),
 nn.Conv1d(32, 32, kernel_size=3, stride=1, padding=1, bias=False),
 nn.BatchNorm1d(32),
 nn.ReLU(inplace=True),
 nn.Conv1d(32, base_filters, kernel_size=3, stride=1, padding=1, bias=False),
 nn.BatchNorm1d(base_filters),
 nn.ReLU(inplace=True),
 )
 self.in_channels = base_filters
 self.layer1 = self._make_layer(64, 3, stride=1)
 self.layer2 = self._make_layer(128, 4, stride=2)
 self.layer3 = self._make_layer(256, 6, stride=2)
 self.layer4 = self._make_layer(512, 3, stride=2)
 self.pool = nn.AdaptiveAvgPool1d(1)
 self.fc = nn.Linear(512 * Bottleneck1D.expansion, num_classes)

 def _make_layer(self, planes, blocks, stride):
 layers = [Bottleneck1D(self.in_channels, planes, stride=stride)]
 self.in_channels = planes * Bottleneck1D.expansion
 for _ in range(1, blocks):
 layers.append(Bottleneck1D(self.in_channels, planes, stride=1))
 return nn.Sequential(*layers)

 def forward(self, signal):
 x = self.stem(signal)
 x = self.layer1(x)
 x = self.layer2(x)
 x = self.layer3(x)
 x = self.layer4(x)
 x = self.pool(x).squeeze(-1)
 logits = self.fc(x).squeeze(1)
 return logits

In [ ]:
# Baseline: real-only XResNet — repeat over TRAIN_SEEDS, same subset (see Train_custum multi-seed cell)
import random

TRAIN_SEEDS = [0, 1, 2]
EPOCHS = 40
LR = 1e-3
VERBOSE_EPOCHS = False
SAVE_CKPT_PER_TRAIN_SEED = False

female_test_mask = df_test_real_mem["sex_binary"].to_numpy() == 1.0
male_test_mask = df_test_real_mem["sex_binary"].to_numpy() == 0.0

def subgroup_metrics(targets, probs, mask):
 t = targets[mask]
 p = probs[mask]
 return {
 "n": len(t),
 "auroc": roc_auc_score(t, p),
 "auprc": average_precision_score(t, p),
 "prevalence": float(np.mean(t)),
 }

def set_train_seed(seed: int):
 random.seed(seed)
 np.random.seed(seed)
 torch.manual_seed(seed)
 if torch.cuda.is_available():
 torch.cuda.manual_seed_all(seed)

seed_results = []
model = None

for train_seed in TRAIN_SEEDS:
 print(f"\n========== TRAIN_SEED {train_seed} ==========")
 set_train_seed(train_seed)
 train_real_loader = make_train_real_loader(train_seed)
 model = XResNet1d50BinaryMIClassifier(in_channels=12).to(device)
 model = fit_model(
 model,
 train_real_loader,
 val_loader,
 device,
 epochs=EPOCHS,
 lr=LR,
 verbose=VERBOSE_EPOCHS,
 )
 test_metrics = evaluate(model, test_loader, device)
 fem = subgroup_metrics(test_metrics["targets"], test_metrics["probs"], female_test_mask)
 mal = subgroup_metrics(test_metrics["targets"], test_metrics["probs"], male_test_mask)
 print(
 f"TEST train_seed={train_seed} | AUROC={test_metrics['auroc']:.4f} "
 f"AUPRC={test_metrics['auprc']:.4f} | F_AUROC={fem['auroc']:.4f} M_AUROC={mal['auroc']:.4f}"
 )
 if SAVE_CKPT_PER_TRAIN_SEED:
 out_dir = DRIVE_SUBSET_DIR
 out_dir.mkdir(parents=True, exist_ok=True)
 ckpt = out_dir / f"xresnet_baseline_subset{SUBSET_SEED}_trainseed{train_seed}.pt"
 torch.save(model.state_dict(), ckpt)
 print("Saved", ckpt)
 seed_results.append(
 {
 "train_seed": int(train_seed),
 "test_loss": float(test_metrics["loss"]),
 "test_auroc": float(test_metrics["auroc"]),
 "test_auprc": float(test_metrics["auprc"]),
 "female_auroc": float(fem["auroc"]),
 "female_auprc": float(fem["auprc"]),
 "male_auroc": float(mal["auroc"]),
 "male_auprc": float(mal["auprc"]),
 }
 )

xresnet_baseline = model
test_metrics = evaluate(xresnet_baseline, test_loader, device)

def _mean_std(key):
 a = np.array([r[key] for r in seed_results], dtype=np.float64)
 m = float(a.mean())
 s = float(a.std(ddof=1)) if len(a) > 1 else 0.0
 return m, s

print("\n========== AGGREGATE over TRAIN_SEEDS (mean ± std) ==========")
for key in [
 "test_auroc",
 "test_auprc",
 "female_auroc",
 "female_auprc",
 "male_auroc",
 "male_auprc",
]:
 m, s = _mean_std(key)
 if len(seed_results) > 1:
 print(f"{key}: {m:.4f} ± {s:.4f}")
 else:
 print(f"{key}: {m:.4f}")

print("\nLast TRAIN_SEED model metrics (same as final test_metrics):")
print({k: v for k, v in test_metrics.items() if k not in ["probs", "targets"]})

In [ ]:
# Per-seed table (aggregates printed in previous cell)
import pandas as pd

pd.DataFrame(seed_results)

## Real + synthetic training (random mixture each step)

**α** = P(real) each sampled step; **1−α** synthetic (`ConcatDataset` + `WeightedRandomSampler`).

- **`ALPHAS`**: list of α values to try (e.g. `[0.5, 0.25]`).
- **`N_MIX_SEEDS`**: number of train seeds (default **3**); seeds are `0 … N_MIX_SEEDS−1`. For each α, metrics are **averaged over seeds** (mean ± std printed; table `mix_summary_by_alpha`).
- **`VERBOSE_MIX_EPOCHS`**: if `True`, prints train/val loss every epoch for every (α, seed) run (very verbose).
- Synthetic files: four strata `male_non_mi_*`, `male_mi_*`, `female_non_mi_*`, `female_mi_*` under **`SYN_SAVE_DIR`**.
- Each epoch uses **`len(train_real_ds)`** samples ⇒ same batches/epoch as the real baseline.
- **`EPOCHS_MIX`**: set equal to baseline **`EPOCHS`**.


In [ ]:
# Real + synthetic: P(real)=alpha per draw; loop ALPHAS × seeds; report mean ± std per alpha
import inspect
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, ConcatDataset, WeightedRandomSampler, DataLoader

# Folder with male_non_mi_*, male_mi_*, female_non_mi_*, female_mi_* (see Generate_ECGs_on_skewed_sex.ipynb)
SYN_SAVE_DIR = Path("/content/drive/MyDrive/sex_mi_synthetic_skewed_25f_75m_even4")
GROUP_NAMES = ["male_non_mi", "male_mi", "female_non_mi", "female_mi"]

_data_chunks, _cond_chunks = [], []
for name in GROUP_NAMES:
 dp = SYN_SAVE_DIR / f"{name}_data.npy"
 cp = SYN_SAVE_DIR / f"{name}_cond.npy"
 if not dp.is_file() or not cp.is_file():
 raise FileNotFoundError(
 f"Expected {dp.name} and {cp.name} under {SYN_SAVE_DIR} (per-group saves from generation notebook)."
 )
 _data_chunks.append(np.load(dp))
 _cond_chunks.append(np.load(cp))

syn_data = np.concatenate(_data_chunks, axis=0).astype(np.float32)
syn_cond = np.concatenate(_cond_chunks, axis=0).astype(np.float32)
syn_mi = syn_cond[:, 1].astype(np.float32)

class SyntheticECGMIDataset(Dataset):
 def __init__(self, signals, mi_labels, cond=None):
 self.signals = np.asarray(signals, dtype=np.float32)
 self.labels = np.asarray(mi_labels, dtype=np.float32)
 self.cond = None if cond is None else np.asarray(cond, dtype=np.float32)

 def __len__(self):
 return len(self.signals)

 def __getitem__(self, idx):
 out = {
 "signal": torch.tensor(self.signals[idx], dtype=torch.float32),
 "label": torch.tensor(self.labels[idx], dtype=torch.float32),
 }
 if self.cond is not None:
 out["cond"] = torch.tensor(self.cond[idx], dtype=torch.float32)
 return out

train_syn_ds = SyntheticECGMIDataset(syn_data, syn_mi, syn_cond)

R = len(train_real_ds)
S = len(train_syn_ds)
print("real train:", R, "| synthetic train:", S)

# Training hyperparameters
EPOCHS_MIX = 75 # set equal to EPOCHS in the real-only baseline cell
LR_MIX = 1e-3

# Try several mixture weights; P(real)=alpha each sampled step
ALPHAS = [0.5, 0.25]

# Average over this many independent train seeds (init + shuffle)
N_MIX_SEEDS = 3
MIX_TRAIN_SEEDS = list(range(N_MIX_SEEDS))

# Per-epoch logs are very long for many (alpha, seed) runs
VERBOSE_MIX_EPOCHS = False

num_samples_per_epoch = R
baseline_batches = (R + batch_size - 1) // batch_size
print(
 "Mixture epoch: num_samples_per_epoch =", num_samples_per_epoch,
 "| batches/epoch (expect match baseline):", baseline_batches,
)

def make_mix_train_loader(alpha: float, train_seed: int) -> DataLoader:
 eps = 1e-10
 a = float(np.clip(alpha, eps, 1.0 - eps))
 combined = ConcatDataset([train_real_ds, train_syn_ds])
 w = np.concatenate(
 [
 np.full(R, (a / R), dtype=np.float64),
 np.full(S, ((1.0 - a) / S), dtype=np.float64),
 ]
 )
 g = torch.Generator()
 g.manual_seed(int(train_seed))
 sampler = WeightedRandomSampler(
 weights=torch.as_tensor(w, dtype=torch.double),
 num_samples=int(num_samples_per_epoch),
 replacement=False,
 generator=g,
 )
 loader = DataLoader(
 combined,
 batch_size=batch_size,
 sampler=sampler,
 num_workers=2,
 pin_memory=True,
 )
 n_batches = len(loader)
 if n_batches != baseline_batches:
 raise RuntimeError(
 f"Batch count mismatch: mixture len(loader)={n_batches} vs baseline {baseline_batches}"
 )
 return loader

_fit_sig = inspect.signature(fit_model)
_use_log_prefix = "log_prefix" in _fit_sig.parameters
if not _use_log_prefix:
 print(
 "fit_model has no log_prefix — re-run the training-helpers cell for prefixed epoch logs."
 )

metric_keys = [
 "test_loss",
 "test_auroc",
 "test_auprc",
 "female_auroc",
 "female_auprc",
 "male_auroc",
 "male_auprc",
]

mix_all_results = []
xresnet_mix = None

for alpha in ALPHAS:
 print(f"\n{'#' * 20} ALPHA={alpha} (P real={alpha}, P synth={1 - alpha}) {'#' * 20}")
 seed_rows = []
 for mix_seed in MIX_TRAIN_SEEDS:
 print(f" --- mix_train_seed={mix_seed} ---")
 set_train_seed(mix_seed)
 train_mix_loader = make_mix_train_loader(alpha, mix_seed)
 xresnet_mix = XResNet1d50BinaryMIClassifier(in_channels=12).to(device)
 log_p = f"[alpha={alpha} seed={mix_seed}] "
 if _use_log_prefix:
 xresnet_mix = fit_model(
 xresnet_mix,
 train_mix_loader,
 val_loader,
 device,
 epochs=EPOCHS_MIX,
 lr=LR_MIX,
 verbose=VERBOSE_MIX_EPOCHS,
 log_prefix=log_p,
 )
 else:
 xresnet_mix = fit_model(
 xresnet_mix,
 train_mix_loader,
 val_loader,
 device,
 epochs=EPOCHS_MIX,
 lr=LR_MIX,
 verbose=VERBOSE_MIX_EPOCHS,
 )
 tm = evaluate(xresnet_mix, test_loader, device)
 fem = subgroup_metrics(tm["targets"], tm["probs"], female_test_mask)
 mal = subgroup_metrics(tm["targets"], tm["probs"], male_test_mask)
 row = {
 "alpha": float(alpha),
 "mix_train_seed": int(mix_seed),
 "epochs": int(EPOCHS_MIX),
 "samples_per_epoch": int(num_samples_per_epoch),
 "batches_per_epoch": int(baseline_batches),
 "test_loss": float(tm["loss"]),
 "test_auroc": float(tm["auroc"]),
 "test_auprc": float(tm["auprc"]),
 "female_auroc": float(fem["auroc"]),
 "female_auprc": float(fem["auprc"]),
 "male_auroc": float(mal["auroc"]),
 "male_auprc": float(mal["auprc"]),
 }
 seed_rows.append(row)
 mix_all_results.append(row)
 print(
 f" {log_p}TEST | loss={row['test_loss']:.4f} AUROC={row['test_auroc']:.4f} "
 f"AUPRC={row['test_auprc']:.4f} | F_AUROC={row['female_auroc']:.4f} "
 f"M_AUROC={row['male_auroc']:.4f}"
 )

 print(f"\n >>> Mean ± std over {len(seed_rows)} seeds for alpha={alpha}")
 for k in metric_keys:
 a = np.array([r[k] for r in seed_rows], dtype=np.float64)
 m, s = float(a.mean()), float(a.std(ddof=1)) if len(a) > 1 else 0.0
 if len(a) > 1:
 print(f" {k}: {m:.4f} ± {s:.4f}")
 else:
 print(f" {k}: {m:.4f}")

def _summarize_alpha(alpha_val, rows):
 out = {"alpha": float(alpha_val), "n_seeds": len(rows)}
 for k in metric_keys:
 a = np.array([r[k] for r in rows], dtype=np.float64)
 out[f"{k}_mean"] = float(a.mean())
 out[f"{k}_std"] = float(a.std(ddof=1)) if len(a) > 1 else 0.0
 return out

mix_summary_by_alpha = [_summarize_alpha(a, [r for r in mix_all_results if r["alpha"] == a]) for a in ALPHAS]

print("\n" + "=" * 60)
print("SUMMARY TABLE: average metrics per alpha (over seeds)")
print("=" * 60)
import pandas as pd
from IPython.display import display

summary_df = pd.DataFrame(mix_summary_by_alpha)
display(summary_df)

# Also show one row per (alpha, seed)
detail_df = pd.DataFrame(mix_all_results)
display(detail_df)


## ResNet + multi-head attention (binary MI)

Same **data protocol** as XResNet: skewed real subset, optional **real+synthetic** mixture. Signal is **`[B, 12, 1000]`** → reshaped to **`[B, 12, 1, 1000]`** for the 2D conv stack. The extra **`l`** vector is **per-lead temporal mean** (shape **`[B, 12]`**), matching the original **`Linear(256 + 12, nOUT)`** design.

The core **`ResnetAttention`** forward **always returns logits** (no sigmoid) so **`BCEWithLogitsLoss`** and **`evaluate()`** stay correct.

Run **baseline attention** after DataLoaders + `fit_model` exist. For **mixture attention**, run the **synthetic load** in the XResNet mixture cell first so **`train_syn_ds`** exists, or run the mixture-attention cell which reloads synth data if missing.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MyResidualBlock(nn.Module):
 def __init__(self, downsample):
 super().__init__()
 self.downsample = downsample
 self.stride = 2 if self.downsample else 1
 K = 9
 P = (K - 1) // 2
 self.conv1 = nn.Conv2d(
 in_channels=256,
 out_channels=256,
 kernel_size=(1, K),
 stride=(1, self.stride),
 padding=(0, P),
 bias=False,
 )
 self.bn1 = nn.BatchNorm2d(256)
 self.conv2 = nn.Conv2d(
 in_channels=256,
 out_channels=256,
 kernel_size=(1, K),
 padding=(0, P),
 bias=False,
 )
 self.bn2 = nn.BatchNorm2d(256)
 if self.downsample:
 self.idfunc_0 = nn.AvgPool2d(kernel_size=(1, 2), stride=(1, 2))
 self.idfunc_1 = nn.Conv2d(
 in_channels=256, out_channels=256, kernel_size=(1, 1), bias=False
 )

 def forward(self, x):
 identity = x
 x = F.leaky_relu(self.bn1(self.conv1(x)))
 x = F.leaky_relu(self.bn2(self.conv2(x)))
 if self.downsample:
 identity = self.idfunc_0(identity)
 identity = self.idfunc_1(identity)
 # Odd time lengths: strided conv and AvgPool can differ by 1 on width
 if identity.shape[-1] != x.shape[-1]:
 identity = F.interpolate(
 identity,
 size=(identity.shape[2], x.shape[-1]),
 mode="nearest",
 )
 return x + identity

class ResnetAttention(nn.Module):
 """nOUT=1 for binary MI; forward returns logits (B,) for BCEWithLogitsLoss."""

 def __init__(self, nOUT=1):
 super().__init__()
 self.conv = nn.Conv2d(
 in_channels=12,
 out_channels=256,
 kernel_size=(1, 15),
 padding=(0, 7),
 stride=(1, 2),
 bias=False,
 )
 self.bn = nn.BatchNorm2d(256)
 self.rb_0 = MyResidualBlock(downsample=True)
 self.rb_1 = MyResidualBlock(downsample=True)
 self.rb_2 = MyResidualBlock(downsample=True)
 self.rb_3 = MyResidualBlock(downsample=True)
 self.rb_4 = MyResidualBlock(downsample=True)
 self.mha = nn.MultiheadAttention(256, 8)
 self.pool = nn.AdaptiveMaxPool1d(output_size=1)
 self.fc_1 = nn.Linear(256 + 12, nOUT)

 def forward(self, x, l):
 # x: (B, 12, 1, T), l: (B, 12)
 x = F.leaky_relu(self.bn(self.conv(x)))
 x = self.rb_0(x)
 x = self.rb_1(x)
 x = self.rb_2(x)
 x = self.rb_3(x)
 x = self.rb_4(x)
 x = F.dropout(x, p=0.5, training=self.training)
 x = x.squeeze(2).permute(2, 0, 1)
 x, _ = self.mha(x, x, x)
 x = x.permute(1, 2, 0)
 x = self.pool(x).squeeze(2)
 x = torch.cat((x, l), dim=1)
 x = self.fc_1(x)
 return x.squeeze(-1)

class ResnetAttentionBinaryMI(nn.Module):
 """ECG (B, 12, T) -> logits (B,)."""

 def __init__(self):
 super().__init__()
 self.backbone = ResnetAttention(nOUT=1)

 def forward(self, signal):
 x = signal.unsqueeze(2)
 l = signal.mean(dim=-1)
 return self.backbone(x, l)

_g = torch.Generator().manual_seed(0)
_x = torch.randn(2, 12, 1000, generator=_g)
_m = ResnetAttentionBinaryMI()
assert _m(_x).shape == (2,), _m(_x).shape
print("ResnetAttentionBinaryMI I/O check OK (2, 12, 1000) -> (2,)")



In [ ]:
# Baseline: ResNet+Attention on real only (multi-seed); same loaders as XResNet baseline
import numpy as np

ATTN_EPOCHS = 75
ATTN_LR = 1e-3
ATTN_VERBOSE = False
ATTN_TRAIN_SEEDS = [0, 1, 2, 3, 4]

attn_baseline_results = []
attn_baseline_model = None

for train_seed in ATTN_TRAIN_SEEDS:
 print(f"\n========== [ATTN] TRAIN_SEED {train_seed} ==========")
 set_train_seed(train_seed)
 train_real_loader = make_train_real_loader(train_seed)
 attn_baseline_model = ResnetAttentionBinaryMI().to(device)
 attn_baseline_model = fit_model(
 attn_baseline_model,
 train_real_loader,
 val_loader,
 device,
 epochs=ATTN_EPOCHS,
 lr=ATTN_LR,
 verbose=ATTN_VERBOSE,
 )
 tm = evaluate(attn_baseline_model, test_loader, device)
 fem = subgroup_metrics(tm["targets"], tm["probs"], female_test_mask)
 mal = subgroup_metrics(tm["targets"], tm["probs"], male_test_mask)
 print(
 f"[ATTN] TEST seed={train_seed} | AUROC={tm['auroc']:.4f} AUPRC={tm['auprc']:.4f} | "
 f"F_AUROC={fem['auroc']:.4f} M_AUROC={mal['auroc']:.4f}"
 )
 attn_baseline_results.append(
 {
 "train_seed": int(train_seed),
 "test_loss": float(tm["loss"]),
 "test_auroc": float(tm["auroc"]),
 "test_auprc": float(tm["auprc"]),
 "female_auroc": float(fem["auroc"]),
 "female_auprc": float(fem["auprc"]),
 "male_auroc": float(mal["auroc"]),
 "male_auprc": float(mal["auprc"]),
 }
 )

attn_baseline_summary = {
 "model": "ResnetAttentionBinaryMI",
 "epochs": ATTN_EPOCHS,
 "seeds": ATTN_TRAIN_SEEDS,
}
for key in [
 "test_auroc",
 "test_auprc",
 "female_auroc",
 "female_auprc",
 "male_auroc",
 "male_auprc",
]:
 a = np.array([r[key] for r in attn_baseline_results], dtype=np.float64)
 attn_baseline_summary[f"{key}_mean"] = float(a.mean())
 attn_baseline_summary[f"{key}_std"] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print("\n========== [ATTN] BASELINE aggregate over seeds ==========")
for key in [
 "test_auroc",
 "test_auprc",
 "female_auroc",
 "female_auprc",
 "male_auroc",
 "male_auprc",
]:
 print(f" {key}: {attn_baseline_summary[key + '_mean']:.4f} ± {attn_baseline_summary[key + '_std']:.4f}")

import pandas as pd
from IPython.display import display

display(pd.DataFrame(attn_baseline_results))



In [ ]:
# Mixture: ResNet+Attention with same P(real)=alpha sampler as XResNet mixture cell
import inspect
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import ConcatDataset, WeightedRandomSampler, DataLoader

if "train_syn_ds" not in globals():
 _syn_dir = Path("/content/drive/MyDrive/sex_mi_synthetic_skewed_25f_75m_even4")
 _groups = ["male_non_mi", "male_mi", "female_non_mi", "female_mi"]
 _dc, _cc = [], []
 for _name in _groups:
 _dp = _syn_dir / f"{_name}_data.npy"
 _cp = _syn_dir / f"{_name}_cond.npy"
 if not _dp.is_file() or not _cp.is_file():
 raise FileNotFoundError(f"Need {_dp} and {_cp} (or run mixture cell above first)")
 _dc.append(np.load(_dp))
 _cc.append(np.load(_cp))
 _sd = np.concatenate(_dc, axis=0).astype(np.float32)
 _sc = np.concatenate(_cc, axis=0).astype(np.float32)
 _smi = _sc[:, 1].astype(np.float32)
 train_syn_ds = SyntheticECGMIDataset(_sd, _smi, _sc)

ATTN_MIX_EPOCHS = 75
ATTN_MIX_LR = 1e-3
ATTN_MIX_VERBOSE = False
ATTN_ALPHAS = [0.5, 0.75]
ATTN_MIX_SEEDS = list(range(5))

R_a = len(train_real_ds)
S_a = len(train_syn_ds)
num_samples_per_epoch_a = R_a
baseline_batches_a = (R_a + batch_size - 1) // batch_size

def make_mix_train_loader_attn(alpha: float, train_seed: int):
 eps = 1e-10
 a = float(np.clip(alpha, eps, 1.0 - eps))
 combined = ConcatDataset([train_real_ds, train_syn_ds])
 w = np.concatenate(
 [
 np.full(R_a, (a / R_a), dtype=np.float64),
 np.full(S_a, ((1.0 - a) / S_a), dtype=np.float64),
 ]
 )
 g = torch.Generator()
 g.manual_seed(int(train_seed))
 sampler = WeightedRandomSampler(
 weights=torch.as_tensor(w, dtype=torch.double),
 num_samples=int(num_samples_per_epoch_a),
 replacement=False,
 generator=g,
 )
 loader = DataLoader(
 combined,
 batch_size=batch_size,
 sampler=sampler,
 num_workers=2,
 pin_memory=True,
 )
 if len(loader) != baseline_batches_a:
 raise RuntimeError(
 f"Batch mismatch: {len(loader)} vs {baseline_batches_a}"
 )
 return loader

_fit_sig_a = inspect.signature(fit_model)
_use_lp_a = "log_prefix" in _fit_sig_a.parameters

_attn_metric_keys = [
 "test_loss",
 "test_auroc",
 "test_auprc",
 "female_auroc",
 "female_auprc",
 "male_auroc",
 "male_auprc",
]

attn_mix_all_results = []
attn_mix_model = None

for alpha in ATTN_ALPHAS:
 print(f"\n{'#' * 15} [ATTN] ALPHA={alpha} {'#' * 15}")
 seed_rows = []
 for mix_seed in ATTN_MIX_SEEDS:
 print(f" --- mix_seed={mix_seed} ---")
 set_train_seed(mix_seed)
 train_mix_ldr = make_mix_train_loader_attn(alpha, mix_seed)
 attn_mix_model = ResnetAttentionBinaryMI().to(device)
 log_p = f"[ATTN alpha={alpha} s={mix_seed}] "
 if _use_lp_a:
 attn_mix_model = fit_model(
 attn_mix_model,
 train_mix_ldr,
 val_loader,
 device,
 epochs=ATTN_MIX_EPOCHS,
 lr=ATTN_MIX_LR,
 verbose=ATTN_MIX_VERBOSE,
 log_prefix=log_p,
 )
 else:
 attn_mix_model = fit_model(
 attn_mix_model,
 train_mix_ldr,
 val_loader,
 device,
 epochs=ATTN_MIX_EPOCHS,
 lr=ATTN_MIX_LR,
 verbose=ATTN_MIX_VERBOSE,
 )
 tm = evaluate(attn_mix_model, test_loader, device)
 fem = subgroup_metrics(tm["targets"], tm["probs"], female_test_mask)
 mal = subgroup_metrics(tm["targets"], tm["probs"], male_test_mask)
 row = {
 "alpha": float(alpha),
 "mix_train_seed": int(mix_seed),
 "test_loss": float(tm["loss"]),
 "test_auroc": float(tm["auroc"]),
 "test_auprc": float(tm["auprc"]),
 "female_auroc": float(fem["auroc"]),
 "female_auprc": float(fem["auprc"]),
 "male_auroc": float(mal["auroc"]),
 "male_auprc": float(mal["auprc"]),
 }
 seed_rows.append(row)
 attn_mix_all_results.append(row)
 print(
 f" {log_p}TEST | AUROC={row['test_auroc']:.4f} AUPRC={row['test_auprc']:.4f} | "
 f"F={row['female_auroc']:.4f} M={row['male_auroc']:.4f}"
 )
 print(f"\n [ATTN] mean ± std over seeds for alpha={alpha}")
 for k in _attn_metric_keys:
 a = np.array([r[k] for r in seed_rows], dtype=np.float64)
 m, s = float(a.mean()), float(a.std(ddof=1)) if len(a) > 1 else 0.0
 print(f" {k}: {m:.4f} ± {s:.4f}" if len(a) > 1 else f" {k}: {m:.4f}")

def _summarize_attn_alpha(av, rows):
 out = {"alpha": float(av), "n_seeds": len(rows)}
 for k in _attn_metric_keys:
 a = np.array([r[k] for r in rows], dtype=np.float64)
 out[f"{k}_mean"] = float(a.mean())
 out[f"{k}_std"] = float(a.std(ddof=1)) if len(a) > 1 else 0.0
 return out

attn_mix_summary_by_alpha = [
 _summarize_attn_alpha(a, [r for r in attn_mix_all_results if r["alpha"] == a])
 for a in ATTN_ALPHAS
]

print("\n" + "=" * 50)
print("[ATTN] SUMMARY per alpha")
print("=" * 50)
import pandas as pd
from IPython.display import display

display(pd.DataFrame(attn_mix_summary_by_alpha))
display(pd.DataFrame(attn_mix_all_results))

